# 15. Imbalanced Image Classification

Study what happens when some classes have fewer training images. We create a small imbalanced CIFAR-10 training set from the local data, train a simple baseline, then train again with class-weighted loss and compare per-class performance.

## 1. Why accuracy can be misleading

Accuracy counts every test image equally. If one class has many more examples, a model can score well by predicting that common class often while missing rare classes. Per-class recall and macro F1 give every class a more equal voice.


In [1]:
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, f1_score, recall_score
torch.manual_seed(42)
data = Path("data")
if not (data / "cifar-10-batches-py").is_dir(): raise FileNotFoundError("Local CIFAR-10 data is missing.")
prep = transforms.Compose([transforms.ToTensor()])
train = datasets.CIFAR10(data, train=True, download=False, transform=prep)
test = datasets.CIFAR10(data, train=False, download=False, transform=prep)
counts = [1000, 500, 250, 125, 60, 60, 60, 60, 60, 60]
labels = np.array(train.targets)
ids = np.concatenate([np.where(labels == c)[0][:n] for c, n in enumerate(counts)])
train_loader = DataLoader(Subset(train, ids), batch_size=64, shuffle=True)
test_loader = DataLoader(test, batch_size=256)
print("Training images per class:", dict(zip(train.classes, counts)))

Training images per class: {'airplane': 1000, 'automobile': 500, 'bird': 250, 'cat': 125, 'deer': 60, 'dog': 60, 'frog': 60, 'horse': 60, 'ship': 60, 'truck': 60}


## 2. Class weights and weighted loss

Class weights increase the penalty for mistakes on underrepresented classes. A common simple choice is inverse frequency: fewer training examples means a larger loss weight. We compare standard loss with this weighted loss while keeping the data and model the same.


In [2]:
def make_model():
    return nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        nn.Flatten(), nn.Linear(32, 10))
def fit(weighted=False):
    torch.manual_seed(42)
    model = make_model()
    weights = torch.tensor(sum(counts) / (10 * np.array(counts)), dtype=torch.float32) if weighted else None
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(3):
        for x, y in train_loader:
            optimizer.zero_grad(); loss_fn(model(x), y).backward(); optimizer.step()
    return model.eval()
baseline, weighted_model = fit(), fit(weighted=True)
print("Both models trained on the same imbalanced training images.")

Both models trained on the same imbalanced training images.


## 3. Oversampling and balanced sampling

- **Oversampling:** repeat minority-class images so those classes appear more often during training.
- **Balanced sampling:** choose a similar number of examples from each class in every batch or epoch. A weighted sampler can do this without copying files.

These methods change which images the model sees; class-weighted loss instead changes the penalty for each mistake. Usually try one approach at a time before combining them.


## 4. Augmentation-based balancing

Create extra varied training examples for minority classes using safe transformations such as horizontal flips or small crops. Augmentation changes the image, unlike simply repeating it. Apply augmentation to training images only—never to validation or test images.


## 5. Compare before and after

Evaluate both models on the same untouched, balanced test set. Overall accuracy shows total correctness; macro F1 and per-class recall reveal whether minority-class performance improved. Results can vary because this is a small, short training demonstration.


In [3]:
def evaluate(model):
    actual, predicted = [], []
    with torch.inference_mode():
        for x, y in test_loader:
            actual.extend(y.numpy()); predicted.extend(model(x).argmax(1).numpy())
    return np.array(actual), np.array(predicted)
actual, pred_base = evaluate(baseline)
_, pred_weighted = evaluate(weighted_model)
rows = []
for name, pred in [("Before: standard loss", pred_base), ("After: weighted loss", pred_weighted)]:
    rows.append({"Model": name, "Accuracy": accuracy_score(actual, pred),
        "Macro F1": f1_score(actual, pred, average="macro"),
        **dict(zip(test.classes, recall_score(actual, pred, average=None)))})
display(pd.DataFrame(rows).set_index("Model").round(3))

,Accuracy,Macro F1,airplane,automobile,bird,cat,deer,dog,frog,horse,ship,truck
Model,,,,,,,,,,,,
Before: standard loss,0.110,0.032,0.979,0.123,0.000,0.00,0.0,0.000,0.000,0.0,0.0,0.000
After: weighted loss,0.197,0.136,0.377,0.414,0.008,0.33,0.0,0.049,0.704,0.0,0.0,0.086


## Conclusion

When classes are imbalanced, accuracy alone may hide weak minority-class results. Compare macro F1 and each class's recall, then select class weights, oversampling, balanced sampling, or targeted augmentation based on the observed results. Always evaluate on an unchanged test set.